# Shrink Florence-2 with GPTQ — without breaking OCR

**Part 2 of 5** · From full-precision model to a smaller, faster one

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Document-OCR-Pipeline/blob/main/colab/02_ocr_pipeline_quant_A.ipynb)

---

## The problem

Florence-2 is accurate at document OCR — but it is **large** and **slow**.

- Full fp16 weights eat VRAM — hard to fit on smaller GPUs or mobile.
- Running every layer at 16-bit is wasteful — many layers tolerate lower precision.
- Quantizing **everything** to int4 blindly **breaks** text detection.

**Goal of this notebook:** shrink the model ~3–4× while keeping OCR quality close to fp16.

---

## Why we use a 4-phase pipeline (not one-shot quantize)

| Phase | What | Why we do this |
|-------|------|----------------|
| **A — Baseline** | int4 every `nn.Linear` with no planning | Sets a **worst-case floor** — if GPTQ cannot beat this, something is wrong |
| **B — Inventory** | List every `nn.Linear` by size | You cannot quantize smartly without knowing **where the weights live** |
| **C — Sensitivity** | Rank layers by output error on real OCR data | Weight MSE alone lies — some layers **amplify** errors downstream |
| **D — GPTQ plan** | Mixed int4 / int8 / fp16 per layer | Protect fragile layers, GPTQ the rest — **beat Phase A** at similar size |

---

## Key idea: only `nn.Linear` layers

A **Linear layer** is a matrix multiply:

```
output = input × Wᵀ + b
```

- Florence-2 has **hundreds** of these (attention Q/K/V, MLP, output head).
- They hold **~90%+ of model parameters**.
- **Why not quantize everything?** LayerNorm, embeddings, and vision patch layers are numerically fragile — forcing int4 on them destroys OCR. We keep them fp16.

---

## How to run

1. **GPU required** — T4 is enough. Run cells top to bottom.
2. Keep `MIXED_PRECISION = True` in config.
3. **Do not skip Phase A** — you need `baseline_ocr` for the final scorecard.
4. If Colab restarts after install, click **Run all**.

**Before:** [01 — Document OCR Pipeline](01_document_ocr_pipeline.ipynb)  
**After:** [03 — Mobile export](03_ocr_pipeline_mobile.ipynb)


## Step 0 — Install dependencies & set config

### What this cell does
- Installs PyTorch, transformers, matplotlib.
- Pins `transformers==4.49.0` (Florence-2 breaks on newer versions).
- Defines every knob you can tune.

### Why we do this
- **Reproducibility** — same versions = same results across Colab sessions.
- **Florence-2 needs 4.49.x** — the model uses custom code that changed in later releases.
- **Config upfront** — you see every hyperparameter before any code runs.

### Settings explained

| Setting | Default | Why this value |
|---------|---------|----------------|
| `MIXED_PRECISION` | `True` | Auto-pick int4/int8/fp16 per layer instead of one-size-fits-all |
| `FP16_SENSITIVE_PCT` | `15` | Top 15% most fragile layers stay fp16 — protects OCR quality |
| `INT8_MID_PCT` | `35` | Next 35% get int8 — good middle ground between size and accuracy |
| `MAX_CALIB_BATCHES` | `8` | OCR passes to collect activation stats — more = better GPTQ, slower |
| `MIN_PARAMS_TO_QUANT` | `4096` | Skip tiny layers — quantizing them saves almost nothing, adds overhead |
| `ALWAYS_FP16_PATTERNS` | lm_head, embed… | These layers are **always** fp16 — quantizing them breaks text output |

> If Colab restarts after pip install, click **Run all**.


In [ ]:
import os, re, subprocess, sys

MODEL_ID = "microsoft/Florence-2-base-ft"
OCR_PROMPT = "<OCR_WITH_REGION>"

MIXED_PRECISION = True
BITS = 4
MAX_CALIB_BATCHES = 8
MAX_QUANT_LAYERS = None
MAX_ANALYZE_LAYERS = None
FP16_SENSITIVE_PCT = 15
INT8_MID_PCT = 35
MIN_PARAMS_TO_QUANT = 4096
ALWAYS_FP16_PATTERNS = ("lm_head", "embed", "vision", "patch_embed")
GPTQ_BLOCK_SIZE = 128
GPTQ_DAMPING = 0.01

def _pip_version(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                       capture_output=True, text=True, check=False)
    m = re.search(r"^Version: (.+)$", r.stdout, re.M)
    return m.group(1) if m else ""

def ensure_transformers():
    target, ok = "4.49.0", lambda v: v.startswith("4.49")
    ver = _pip_version("transformers")
    if not ok(ver):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
            "--force-reinstall", "transformers==4.49.0"])
        ver = _pip_version("transformers")
    import transformers
    if not ok(transformers.__version__):
        print("Restart runtime, then Run all."); os.kill(os.getpid(), 9)
    return ver

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.26", "torch", "accelerate", "pillow", "matplotlib", "requests", "huggingface_hub"])
print(f"Ready — transformers {ensure_transformers()}")


## Step 1 — Core library (one small cell per topic)

### Why split into small cells?
- **Easier to read** — one idea per cell, not 300 lines at once.
- **Easier to debug** — if something breaks, you know exactly which class failed.
- **Run in order** — each cell depends on the one above it.

### Cell map (run top → bottom)

| # | Topic | Classes |
|---|-------|---------|
| 1a | Imports | `torch`, `DEVICE` |
| 1b | Pack / unpack | `SymmetricQuantizer`, `GPTQState`, `RTNState` |
| 1c | Quant layers | `GPTQLinear`, `RTNLinear` |
| 1d | Algorithms | `RTNQuantizer`, `GPTQQuantizer` |
| 1e | Helpers | `QuantLinearFactory`, `LinearMetrics`, `LinearLayerScanner`, `ModelPatcher` |
| 1f | Calibration | `CalibrationSession` |
| 1g | Compare | `QuantMethodComparator` |


In [ ]:
# 1a — Imports
from __future__ import annotations
import math, time
from abc import ABC
from dataclasses import dataclass
from io import BytesIO

import matplotlib.pyplot as plt
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUANT_MODULE_TYPES = (nn.Linear,)
print(f"Device: {DEVICE}")


#### 1b — `SymmetricQuantizer` + packed states

**What:** round fp16 weights to int4 and store them in a compact `GPTQState` / `RTNState`.

**Why:** every quantizer (RTN and GPTQ) needs the same pack/unpack logic.

In [ ]:
class SymmetricQuantizer:
    def __init__(self, n_bits: int = 4):
        self.n_bits = n_bits

    def qmax(self) -> int:
        return 2 ** (self.n_bits - 1) - 1

    def quantize(self, W: torch.Tensor):
        qmax = self.qmax()
        scales = W.abs().amax(1).clamp(min=1e-8) / qmax
        q = torch.round(W / scales.unsqueeze(1)).clamp(-qmax - 1, qmax).to(torch.int8)
        return q, scales

    def dequantize(self, q, scales):
        return q.float() * scales.unsqueeze(1)


@dataclass
class GPTQState:
    weight_q: torch.Tensor
    weight_scales: torch.Tensor
    bias: torch.Tensor | None
    n_bits: int = 4
    method: str = "gptq"

@dataclass
class RTNState:
    weight_q: torch.Tensor
    weight_scales: torch.Tensor
    bias: torch.Tensor | None
    n_bits: int = 4
    method: str = "rtn"

QuantState = GPTQState | RTNState
print("SymmetricQuantizer + states OK")

#### 1c — Quantized `nn.Linear` modules

**What:** `GPTQLinear` / `RTNLinear` replace a normal `nn.Linear` in the model.

**Why:** swap layers in-place without changing the rest of the forward graph.

In [ ]:
class BaseQuantLinear(nn.Module, ABC):
    def __init__(self, in_f: int, out_f: int, state: QuantState):
        super().__init__()
        self.in_features, self.out_features = in_f, out_f
        self.n_bits, self.method = state.n_bits, state.method
        self.register_buffer("weight_q", state.weight_q)
        self.register_buffer("weight_scales", state.weight_scales)
        self.bias = nn.Parameter(state.bias.clone()) if state.bias is not None else None
        self._sq = SymmetricQuantizer(state.n_bits)

    @property
    def weight_fp(self):
        return self._sq.dequantize(self.weight_q, self.weight_scales)

    def forward(self, x):
        return F.linear(x, self.weight_fp.to(x.dtype), self.bias)

    def storage_bytes(self):
        n = self.weight_q.numel() + self.weight_scales.numel()
        return (n + (self.bias.numel() if self.bias is not None else 0)) * 4

class GPTQLinear(BaseQuantLinear):
    def __init__(self, in_f, out_f, state: GPTQState):
        super().__init__(in_f, out_f, state)

class RTNLinear(BaseQuantLinear):
    def __init__(self, layer: nn.Linear, state: RTNState):
        super().__init__(layer.in_features, layer.out_features, state)

print("GPTQLinear / RTNLinear OK")

#### 1d — Quantization algorithms (`RTNQuantizer`, `GPTQQuantizer`)

**What:**
- `RTNQuantizer` — round-to-nearest (naive, fast).
- `GPTQQuantizer` — Hessian-aware column-wise quant (uses calibration data).

**Why:** RTN is the Phase A baseline; GPTQ is the Phase D solution.

In [ ]:
class RTNQuantizer:
    def __init__(self, layer: nn.Linear, n_bits: int = 4):
        self.layer, self.n_bits = layer, n_bits
        self._sq = SymmetricQuantizer(n_bits)

    def quantize(self) -> RTNState:
        q, s = self._sq.quantize(self.layer.weight.data.float())
        b = self.layer.bias.data.clone() if self.layer.bias is not None else None
        return RTNState(q, s, b, self.n_bits)

class GPTQQuantizer:
    def __init__(self, layer: nn.Linear, n_bits=4, block_size=128, damping=0.01):
        self.layer = layer
        self.n_bits, self.block_size, self.damping = n_bits, block_size, damping
        self.H, self.nsamples = None, 0
        self._sq = SymmetricQuantizer(n_bits)

    def add_batch(self, inp: torch.Tensor):
        if inp.dim() == 3: inp = inp.reshape(-1, inp.shape[-1])
        inp = inp.float()
        if self.H is None:
            self.H = torch.zeros(inp.shape[1], inp.shape[1], device=inp.device)
        self.H += inp.t() @ inp
        self.nsamples += inp.shape[0]

    def quantize(self) -> GPTQState:
        W = self.layer.weight.data.float().clone()
        H = self.H.clone()
        dead = torch.diag(H) == 0
        H[dead, dead] = 1.0; W[:, dead] = 0.0
        H[torch.arange(H.shape[0], device=H.device), torch.arange(H.shape[0], device=H.device)] += self.damping * H.diag().mean()
        H = torch.linalg.cholesky(H)
        Hinv = torch.linalg.cholesky(torch.linalg.cholesky_inverse(H), upper=True)
        Q, qmax = torch.zeros_like(W), self._sq.qmax()
        for i1 in range(0, W.shape[1], self.block_size):
            i2 = min(i1 + self.block_size, W.shape[1])
            W1, Q1, Err1 = W[:, i1:i2].clone(), torch.zeros_like(W[:, i1:i2]), torch.zeros_like(W[:, i1:i2])
            Hinv1 = Hinv[i1:i2, i1:i2]
            for i in range(i2 - i1):
                w, d = W1[:, i], Hinv1[i, i]
                sc = w.abs().max().clamp(min=1e-8) / qmax
                q = torch.round(w / sc).clamp(-qmax - 1, qmax)
                Q1[:, i] = q
                err = (w - q * sc) / d
                W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)
                Err1[:, i] = err
            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]
        wq, ws = self._sq.quantize(Q)
        return GPTQState(wq.cpu(), ws.cpu(),
            self.layer.bias.detach().cpu() if self.layer.bias is not None else None, self.n_bits)

print("RTNQuantizer + GPTQQuantizer OK")

#### 1e — Factory, metrics, scanner, patcher

**What:**
- `QuantLinearFactory` — build the right quant layer from a state.
- `LinearMetrics` — weight MSE and output MSE.
- `LinearLayerScanner` — find all `nn.Linear` in the model.
- `ModelPatcher` — swap one layer by name.

**Why:** shared utilities used in every phase.

In [ ]:
class QuantLinearFactory:
    @staticmethod
    def build(layer: nn.Linear, state: QuantState) -> BaseQuantLinear:
        if isinstance(state, GPTQState):
            return GPTQLinear(layer.in_features, layer.out_features, state)
        return RTNLinear(layer, state)

class LinearMetrics:
    @staticmethod
    def num_params(layer: nn.Linear) -> int:
        return layer.weight.numel() + (layer.bias.numel() if layer.bias is not None else 0)

    @staticmethod
    def weight_mse(orig: nn.Linear, quant: BaseQuantLinear) -> float:
        return (orig.weight.float() - quant.weight_fp).pow(2).mean().item()

    @staticmethod
    def output_mse(orig, quant, inputs) -> float:
        if inputs is None or inputs.numel() == 0: return float("nan")
        x = inputs[:2048].to(orig.weight.device)
        with torch.no_grad():
            y0 = F.linear(x, orig.weight.float(), orig.bias)
            y1 = quant(x)
        return (y0 - y1).pow(2).mean().item()

class LinearLayerScanner:
    def __init__(self, root: nn.Module):
        self.root = root

    def layers(self, limit=None):
        out = [(n, m) for n, m in self.root.named_modules() if isinstance(m, nn.Linear)]
        return out[:limit] if limit else out

    def is_protected(self, name: str) -> bool:
        n = name.lower()
        return any(p in n for p in ALWAYS_FP16_PATTERNS)

class ModelPatcher:
    @staticmethod
    def replace(model, name, mod):
        parent_name, _, child_name = name.rpartition(".")
        parent = model.get_submodule(parent_name) if parent_name else model
        setattr(parent, child_name, mod)

print("Factory + metrics + scanner + patcher OK")

#### 1f — `CalibrationSession`

**What:** run OCR forwards on the calibration image and hook layer inputs.

**Why:** GPTQ needs real activation statistics from your OCR task — not random noise.

In [ ]:
class CalibrationSession:
    def __init__(self, model, processor, image, prompt=OCR_PROMPT, device=DEVICE):
        self.model, self.processor, self.image = model, processor, image
        self.prompt, self.device = prompt, device

    def _padded(self):
        w, h = self.image.size
        side = max(w, h)
        canvas = Image.new("RGB", (side, side), "white")
        px, py = (side - w) // 2, (side - h) // 2
        canvas.paste(self.image, (px, py))
        return canvas

    def run(self, n_batches: int):
        padded = self._padded()
        for _ in range(n_batches):
            inp = self.processor(text=self.prompt, images=padded, return_tensors="pt").to(self.device)
            inp["pixel_values"] = inp["pixel_values"].to(dtype=next(self.model.parameters()).dtype)
            with torch.no_grad():
                self.model.generate(input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
                                    max_new_tokens=64, do_sample=False, num_beams=1)

    def register_gptq_hooks(self, quantizers: dict):
        handles = []
        for name, q in quantizers.items():
            def hook(m, inp, out, quantizer=q):
                x = inp[0] if isinstance(inp, tuple) else inp
                if x is not None: quantizer.add_batch(x.detach())
            handles.append(self.model.get_submodule(name).register_forward_hook(hook))
        return handles

    def collect_inputs(self, layer_names, n_batches):
        store = {n: [] for n in layer_names}
        def make_hook(n):
            def hook(m, inp, out):
                x = inp[0] if isinstance(inp, tuple) else inp
                if x is not None: store[n].append(x.detach().reshape(-1, x.shape[-1]).cpu())
            return hook
        handles = [self.model.get_submodule(n).register_forward_hook(make_hook(n)) for n in layer_names]
        self.run(n_batches)
        for h in handles: h.remove()
        return {n: torch.cat(v, 0) if v else None for n, v in store.items()}

print("CalibrationSession OK")

#### 1g — `QuantMethodComparator` + aliases

**What:** compare RTN vs GPTQ output error on one layer (Step 15).

**Why:** proves GPTQ beats naive rounding on real OCR activations.

In [ ]:
class QuantMethodComparator:
    def __init__(self, cal: CalibrationSession):
        self.cal = cal

    def rtn_mse(self, layer, inputs, bits=4):
        qm = QuantLinearFactory.build(layer, RTNQuantizer(layer, bits).quantize()).to(layer.weight.device)
        return LinearMetrics.output_mse(layer, qm, inputs)

    def gptq_mse(self, layer, inputs, bits=4):
        gq = GPTQQuantizer(layer, bits)
        h = layer.register_forward_hook(lambda m, i, o, q=gq: q.add_batch(i[0].detach()))
        self.cal.run(2)
        h.remove()
        qm = QuantLinearFactory.build(layer, gq.quantize()).to(layer.weight.device)
        return LinearMetrics.output_mse(layer, qm, inputs)

# backward-compat aliases
def build_quantized_linear(layer, state): return QuantLinearFactory.build(layer, state)
build_quantized_module = build_quantized_linear
def iter_linear_layers(root, limit=None): return LinearLayerScanner(root).layers(limit)
iter_linear_modules = iter_quantizable_modules = iter_linear_layers
def layer_num_params(l): return LinearMetrics.num_params(l)
def layer_weight_mse(a, b): return LinearMetrics.weight_mse(a, b)
def layer_output_mse(a, b, c): return LinearMetrics.output_mse(a, b, c)
def replace_module(m, n, mod): ModelPatcher.replace(m, n, mod)
GenericRTNQuantizer = RTNQuantizer

print("QuantMethodComparator + aliases OK")

## Step 2 — Load Florence-2 and calibration image

### Cell map

| # | Topic | What you get |
|---|-------|--------------|
| 2a | Loader classes | `ImagePadder`, `FlorenceModelLoader` |
| 2b | OCR detector | `FlorenceOCRDetector` |
| 2c | Run load | `processor`, `model`, `image` |

### Why we do this
- **Processor** — Florence-2 needs its own tokenizer + image preprocessor.
- **fp16 model** — full-precision reference; only modified in Phase D.
- **Calibration image** — real document page for GPTQ and sensitivity.


In [ ]:
# 2a — Loader classes
ensure_transformers()
from transformers import AutoProcessor, AutoModelForCausalLM

class ImagePadder:
    @staticmethod
    def pad(image):
        w, h = image.size; side = max(w, h)
        c = Image.new("RGB", (side, side), "white")
        px, py = (side-w)//2, (side-h)//2
        c.paste(image, (px, py))
        return c, px, py, w, h

class FlorenceModelLoader:
    def __init__(self, model_id=MODEL_ID, device=DEVICE):
        self.model_id, self.device = model_id, device
        self.dtype = torch.float16 if device == "cuda" else torch.float32

    def load_processor(self):
        return AutoProcessor.from_pretrained(self.model_id, trust_remote_code=True)

    def load_model(self):
        m = AutoModelForCausalLM.from_pretrained(
            self.model_id, trust_remote_code=True,
            torch_dtype=self.dtype, attn_implementation="eager").to(self.device)
        m.eval(); return m

    def fresh_copy(self): return self.load_model()

print("ImagePadder + FlorenceModelLoader OK")


#### 2b — `FlorenceOCRDetector`

**What:** run OCR on an image and return detected text lines + bounding boxes.

**Why:** every phase measures quality by **line count**, not just weight MSE.

In [ ]:
class FlorenceOCRDetector:
    def __init__(self, processor, prompt=OCR_PROMPT):
        self.processor, self.prompt = processor, prompt

    def detect(self, image, model, device, max_new_tokens=512):
        padded, px, py, ow, oh = ImagePadder.pad(image)
        inp = self.processor(text=self.prompt, images=padded, return_tensors="pt").to(device)
        inp["pixel_values"] = inp["pixel_values"].to(dtype=next(model.parameters()).dtype)
        t0 = time.time()
        with torch.no_grad():
            gen = model.generate(input_ids=inp["input_ids"], pixel_values=inp["pixel_values"],
                                 max_new_tokens=max_new_tokens, num_beams=1, use_cache=False)
        elapsed = time.time() - t0
        raw = self.processor.batch_decode(gen, skip_special_tokens=False)[0]
        parsed = self.processor.post_process_generation(raw, task=self.prompt, image_size=(max(ow,oh),)*2)
        region = parsed.get(self.prompt, {})
        lines = []
        for quad, label in zip(region.get("quad_boxes",[]), region.get("labels",[])):
            xs, ys = quad[0::2], quad[1::2]
            b = [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]
            x1,y1 = max(0,min(ow,b[0]-px)), max(0,min(oh,b[1]-py))
            x2,y2 = max(0,min(ow,b[2]-px)), max(0,min(oh,b[3]-py))
            if x2>x1 and y2>y1:
                lines.append({"text": re.sub(r"</?\\w+[^>]*>","",str(label)).strip(),
                              "bbox": [x1,y1,x2,y2]})
        return lines, elapsed

def pad_info(img): return ImagePadder.pad(img)
def run_florence_detect(img, proc, m, dev, max_new_tokens=512):
    return FlorenceOCRDetector(proc).detect(img, m, dev, max_new_tokens)

print("FlorenceOCRDetector OK")

#### 2c — Load model + calibration image

**What:** download Florence-2, load `processor` + `model`, fetch calibration page.

**Why:** these three variables are used in every phase — load once here.

In [ ]:
_loader = FlorenceModelLoader()
processor = _loader.load_processor()
model = _loader.load_model()
dtype = _loader.dtype

def load_fresh_model(): return _loader.fresh_copy()

try:
    url = "https://raw.githubusercontent.com/Gaurav14cs17/Document-OCR-Pipeline/main/assets/table_page.png"
    image = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB")
except Exception:
    image = Image.new("RGB", (640,480), "white")
    ImageDraw.Draw(image).text((20,20), "Sample", fill="black")

print(f"Loaded {MODEL_ID} — {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
print(f"Calibration image: {image.size[0]}×{image.size[1]} px")
plt.figure(figsize=(6,4)); plt.imshow(image); plt.title("Calibration page"); plt.axis("off"); plt.show()

## Phase A — Naive baseline

### The question
> What happens if I quantize **every** `nn.Linear` to int4 with **no planning**?

### Why we do this first
- It is the **worst-case scenario** — a floor that Phase D must beat.
- Proves that blind int4 **does** hurt OCR — motivates the smarter pipeline.
- Saves `baseline_ocr` for the three-way scorecard in Step 14.
- Uses a **throwaway copy** (`model_naive`) so the main `model` stays fp16 for Phases B–C.

### What happens (3 cells)
1. **Define `NaiveQuantBaseline`** — class that RTN-int4's every Linear layer.
2. **Run on Florence-2** — quantize `model_naive`, print compression ratio.
3. **OCR comparison** — fp16 vs naive int4 line count.

### What to look for
- Compression ~3–4× (fp16 → int4).
- **Line count drops** on naive int4 vs fp16 — this is expected and OK.
- Average weight MSE tells you how noisy the naive quant is.


#### Phase A — define `NaiveQuantBaseline`

**What:** RTN int4 every `nn.Linear` with no planning.

**Why:** worst-case floor — Phase D must beat this.

In [ ]:
@dataclass
class NaiveProfile:
    name: str; bits: int; weight_mse: float; fp_bytes: int; q_bytes: int

LayerProfile = NaiveProfile

class NaiveQuantBaseline:
    def __init__(self, n_bits=4, device=DEVICE):
        self.n_bits, self.device = n_bits, device

    def run(self, model, max_layers=MAX_QUANT_LAYERS) -> list[NaiveProfile]:
        profiles = []
        layers = LinearLayerScanner(model).layers(max_layers)
        for i, (name, layer) in enumerate(layers, 1):
            ql = QuantLinearFactory.build(layer, RTNQuantizer(layer, self.n_bits).quantize()).to(self.device)
            profiles.append(NaiveProfile(name, self.n_bits, LinearMetrics.weight_mse(layer, ql),
                LinearMetrics.num_params(layer)*2, ql.storage_bytes()))
            ModelPatcher.replace(model, name, ql)
            if i % 20 == 0 or i == len(layers): print(f"  {i}/{len(layers)} layers")
        return profiles

def apply_naive_uniform_quant(model, n_bits=4):
    return NaiveQuantBaseline(n_bits).run(model)

if __name__ == "__main__":
    tiny = nn.Sequential(nn.Linear(4,2), nn.Linear(2,1)).to(DEVICE)
    print(f"Demo: {len(NaiveQuantBaseline(4).run(tiny))} NaiveProfile rows")


**▶ Run — quantize `model_naive`**

### What happens
- Loads a fresh fp16 copy → `model_naive`.
- RTN int4 on every `nn.Linear` (round-to-nearest, no calibration).

### Why a separate model copy
- Main `model` must stay fp16 for Phases B and C (inventory + sensitivity).
- `model_naive` is throwaway — only used for baseline comparison.


In [ ]:
# Stage 8b — run naive baseline
print("Loading fp16 copy → model_naive")
model_naive = load_fresh_model()
t0 = time.time()
profiles_a = NaiveQuantBaseline(n_bits=4).run(model_naive)
elapsed = time.time() - t0
if profiles_a:
    avg = sum(p.weight_mse for p in profiles_a) / len(profiles_a)
    comp = sum(p.fp_bytes for p in profiles_a) / max(sum(p.q_bytes for p in profiles_a), 1)
    print(f"Done {elapsed:.1f}s | {len(profiles_a)} layers | avg MSE {avg:.2e} | {comp:.1f}× compression")


**▶ Run — OCR comparison (fp16 vs naive int4)**

### What happens
- Runs Florence OCR on fp16 reference and naive int4 `model_naive`.
- Prints line count and inference time for each.

### Why we compare OCR, not just MSE
- Weight MSE does not tell you if text detection still works.
- **Line count** is the real quality metric for this pipeline.
- Saves `baseline_ocr` dict for Step 14 scorecard.


In [ ]:
# Stage 8c — OCR baseline
detector = FlorenceOCRDetector(processor)
ref = load_fresh_model()
fp16_lines, fp16_t = detector.detect(image, ref, DEVICE)
del ref
if DEVICE == "cuda": torch.cuda.empty_cache()
naive_lines, naive_t = detector.detect(image, model_naive, DEVICE)
print(f"{'':12} {'Lines':>6} {'Time':>8}")
print(f"{'fp16':12} {len(fp16_lines):>6} {fp16_t:>8.2f}s")
print(f"{'naive int4':12} {len(naive_lines):>6} {naive_t:>8.2f}s")
print(f"Δ lines: {len(naive_lines)-len(fp16_lines):+d}")
baseline_ocr = {"fp16_lines": len(fp16_lines), "naive_lines": len(naive_lines),
                "fp16_infer_s": fp16_t, "naive_infer_s": naive_t}


## Phase B — Layer inventory

### The question
> Which `nn.Linear` layers are **biggest**? Which are **protected**?

### Why we do this
- You cannot assign int4/int8/fp16 smartly without knowing **where parameters live**.
- A few large layers often hold 50%+ of Linear weights — these dominate compression.
- **Protected layers** (`lm_head`, embeddings, vision) must stay fp16 — forcing int4 breaks text output.
- Builds `profiles_b` used by Phase C and D.

### What happens (2 cells)
1. **Define `LinearInventory`** — scans model, builds profile rows.
2. **Run** — prints top-12 layers + bar chart.

### What to look for
- `[PROTECT]` tag on `lm_head`, `embed`, `vision` layers.
- Top 3–5 layers account for most of the bar chart.
- Total Linear params as % of model — usually 85–95%.


#### Phase B — define `LinearInventory`

**What:** scan model, list every `nn.Linear` by size, flag protected layers.

**Why:** you need to know where weights live before assigning bits.

In [ ]:
@dataclass
class InventoryProfile:
    name: str; module_type: str; shape: tuple; num_params: int
    pct_of_linear: float; protected: bool; tiny: bool

class LinearInventory:
    def __init__(self, model):
        self.model = model
        self.scanner = LinearLayerScanner(model)

    def summary(self):
        layers = self.scanner.layers()
        lp = sum(LinearMetrics.num_params(m) for _, m in layers)
        mp = sum(p.numel() for p in self.model.parameters())
        return {"count": len(layers), "linear_params": lp, "model_params": mp,
                "pct": 100 * lp / max(mp, 1)}

    def build(self, limit=None):
        layers = self.scanner.layers(limit)
        total = sum(LinearMetrics.num_params(m) for _, m in layers)
        rows = [InventoryProfile(n, "Linear", tuple(m.weight.shape), LinearMetrics.num_params(m),
            100*LinearMetrics.num_params(m)/max(total,1), self.scanner.is_protected(n),
            LinearMetrics.num_params(m) < MIN_PARAMS_TO_QUANT) for n, m in layers]
        rows.sort(key=lambda p: p.num_params, reverse=True)
        return rows, total

def build_layer_inventory(layers):
    total = sum(LinearMetrics.num_params(m) for _, m in layers)
    sc = LinearLayerScanner(nn.Linear(1,1))
    rows = [InventoryProfile(n,"Linear",tuple(m.weight.shape),LinearMetrics.num_params(m),
        100*LinearMetrics.num_params(m)/max(total,1), sc.is_protected(n),
        LinearMetrics.num_params(m)<MIN_PARAMS_TO_QUANT) for n,m in layers]
    rows.sort(key=lambda p: p.num_params, reverse=True)
    return rows, total

def linear_layer_summary(model):
    inv = LinearInventory(model)
    return inv.summary()

LayerProfile = InventoryProfile

if __name__ == "__main__":
  inv = LinearInventory(model)
  print("summary:", inv.summary())


**▶ Run — print layer inventory**

### What happens
- Scans all `nn.Linear` in the fp16 `model`.
- Ranks by parameter count, flags protected/tiny layers.
- Draws bar chart of top 12.

### Why on the fp16 model (not model_naive)
- Inventory is about **architecture**, not quantization state.
- `model` is still fp16 here — Phase D hasn't run yet.


In [ ]:
# Stage 9 — inventory
inv = LinearInventory(model)
summary = inv.summary()
layers = LinearLayerScanner(model).layers(MAX_ANALYZE_LAYERS or MAX_QUANT_LAYERS)
profiles_b, _ = build_layer_inventory(layers)
print(f"Linear layers: {summary['count']} ({summary['pct']:.1f}% of model params)\n")
print(f"{'Rank':<5} {'Params':>10}  Name")
for i, r in enumerate(profiles_b[:12], 1):
    tag = " [PROTECT]" if r.protected else ""
    print(f"{i:<5} {r.num_params:>10,}  {r.name.split('.')[-1]}{tag}")
fig, ax = plt.subplots(figsize=(10,4))
show = profiles_b[:12]
ax.barh([p.name.split(".")[-1] for p in show][::-1], [p.pct_of_linear for p in show][::-1])
ax.set_xlabel("% of Linear params"); ax.set_title("Biggest nn.Linear layers")
plt.tight_layout(); plt.show()


## Phase C — Sensitivity ranking

### The question
> If I int4 **this one layer**, how much does its **output** change on real OCR data?

### Why we do this
- **Weight MSE alone is misleading** — a layer can have low weight error but still amplify errors downstream.
- We measure **output MSE** using activations from real OCR forwards.
- High-sensitivity layers → candidates for fp16 or int8 in Phase D.
- Low-sensitivity layers → safe for int4 GPTQ.
- Saves `profiles_c` and `captures` (input tensors per layer).

### How it works
1. **Capture** — `CalibrationSession` runs OCR, hooks each layer's input.
2. **Measure** — quantize each layer (RTN int4/int8), compare output to fp16.
3. **Rank** — sort by sensitivity score (output MSE × activation magnitude).

### What to look for
- Fragile layers at the top of the ranking.
- Protected layers appear but will be skipped in the plan.
- Red chart = sensitivity, blue chart = output MSE @ int4.


#### Phase C — define `SensitivityAnalyzer`

**What:** rank layers by output MSE when naively quantized on real OCR data.

**Why:** weight MSE alone does not predict which layers break OCR.

In [ ]:
@dataclass
class SensitivityProfile:
    name: str; module_type: str; shape: tuple; num_params: int; protected: bool
    weight_mse_int4: float; weight_mse_int8: float
    output_mse_int4: float; output_mse_int8: float; act_max: float; sensitivity: float

class SensitivityAnalyzer:
    def __init__(self, calibrator: CalibrationSession):
        self.cal = calibrator

    def _w_mse(self, layer, bits):
        sq = SymmetricQuantizer(bits)
        q, s = sq.quantize(layer.weight.float())
        return (layer.weight.float() - sq.dequantize(q,s)).pow(2).mean().item()

    def _o_mse(self, layer, inputs, bits):
        if inputs is None or inputs.numel()==0: return float("nan")
        qm = QuantLinearFactory.build(layer, RTNQuantizer(layer,bits).quantize()).to(layer.weight.device)
        return LinearMetrics.output_mse(layer, qm, inputs)

    def analyze(self, profiles_b, layers, captures) -> list[SensitivityProfile]:
        inv = {p.name: p for p in profiles_b}
        rows = []
        for name, layer in layers:
            p = inv[name]; inp = captures.get(name)
            am = float(inp.abs().max()) if inp is not None else 0.
            o4, o8 = self._o_mse(layer, inp, 4), self._o_mse(layer, inp, 8)
            sens = o4 * (1 + 0.1*math.log1p(am)) if not math.isnan(o4) else 0.
            rows.append(SensitivityProfile(name,"Linear",tuple(layer.weight.shape),p.num_params,p.protected,
                self._w_mse(layer,4), self._w_mse(layer,8), o4, o8, am, sens))
        rows.sort(key=lambda r: r.sensitivity, reverse=True)
        return rows

def build_quantizer(layer, n_bits=None):
    return GPTQQuantizer(layer, n_bits or BITS, GPTQ_BLOCK_SIZE, GPTQ_DAMPING)
def build_quantizer_for_module(layer, n_bits=None): return build_quantizer(layer, n_bits)
def analyze_sensitivity(pb, layers, cap):
    return SensitivityAnalyzer(CalibrationSession(model,processor,image,OCR_PROMPT)).analyze(pb,layers,cap)
def run_calibration(m,p,img,prompt,n):
    CalibrationSession(m,p,img,prompt).run(n)
def register_calibration_hooks(m,q):
    return CalibrationSession(m,processor,image,OCR_PROMPT).register_gptq_hooks(q)
def collect_layer_inputs(m,names,p,img,prompt,nb):
    return CalibrationSession(m,p,img,prompt).collect_inputs(names,nb)

LayerProfile = SensitivityProfile

if __name__ == "__main__":
    print("SensitivityAnalyzer ready — run Stage 10 cell")


**▶ Run — capture activations + rank sensitivity**

### What happens
- Runs `MAX_CALIB_BATCHES` OCR forwards on the calibration image.
- Hooks every `nn.Linear` input tensor → `captures`.
- Ranks layers by sensitivity score.

### Why real OCR forwards (not random data)
- GPTQ and sensitivity need activations from **your actual task**.
- Random Gaussian inputs give wrong Hessian estimates → bad quantization.
- Same image as Phase A/D ensures fair comparison.


In [ ]:
# Stage 10 — sensitivity
cal = CalibrationSession(model, processor, image, OCR_PROMPT)
layers = LinearLayerScanner(model).layers(MAX_ANALYZE_LAYERS or MAX_QUANT_LAYERS)
names = [n for n,_ in layers]
print(f"Capturing inputs for {len(names)} layers...")
captures = cal.collect_inputs(names, MAX_CALIB_BATCHES)
profiles_c = SensitivityAnalyzer(cal).analyze(profiles_b, layers, captures)
print(f"\n{'Rank':<5} {'Sensitivity':>11}  Layer")
for i,p in enumerate(profiles_c[:12],1):
    print(f"{i:<5} {p.sensitivity:>11.2e}  {p.name.split('.')[-1]}{' [PROTECT]' if p.protected else ''}")
fig, ax = plt.subplots(1,2, figsize=(14,5))
show = profiles_c[:15]
lbl = [p.name.split(".")[-1] for p in show]
ax[0].barh(lbl[::-1], [p.sensitivity for p in show][::-1], color="#e74c3c")
ax[0].set_title("Sensitivity (higher = more fragile)")
ax[1].barh(lbl[::-1], [p.output_mse_int4 for p in show][::-1], color="#3498db")
ax[1].set_title("Output MSE @ int4")
plt.tight_layout(); plt.show()


## Phase D — GPTQ mixed precision

### The question
> Can GPTQ beat the Phase A naive baseline at similar model size?

### Why we do this
- Phase A proved blind int4 hurts OCR — now we fix it.
- **Mixed precision** keeps fragile layers at fp16, medium at int8, rest at int4.
- **GPTQ** (not RTN) uses calibration Hessian to minimize output error per column.
- This is the only phase that **modifies the main `model`**.

### The plan (4 sub-steps)
1. **`QuantPlanBuilder`** — assign bits per layer from sensitivity rank
   - Top 15% sensitive → **fp16** (protected + high-sensitivity)
   - Next 35% → **int8 GPTQ**
   - Rest → **int4 GPTQ**
2. **Apply GPTQ** — calibrate + swap layers to `GPTQLinear`
3. **OCR verify** — run detection on quantized model
4. **Scorecard** — fp16 vs naive int4 vs GPTQ mixed

### What to look for
- Pie chart: param share across int4 / int8 / fp16.
- Weight MSE chart: int8 layers should have lower error than int4.
- **GPTQ line count ≥ naive int4** = success.


In [ ]:
# Phase D — QuantPlanBuilder
@dataclass
class PlanProfile:
    name: str; module_type: str; num_params: int; sensitivity: float; protected: bool
    bits: str = "int4"; note: str = ""
    weight_mse: float = 0.; output_mse: float = 0.
    fp_bytes: int = 0; q_bytes: int = 0; applied: bool = False

class QuantPlanBuilder:
    def __init__(self, fp16_pct=FP16_SENSITIVE_PCT, int8_pct=INT8_MID_PCT):
        self.fp16_pct, self.int8_pct = fp16_pct, int8_pct

    def build_mixed(self, profiles_c) -> list[PlanProfile]:
        plan = [PlanProfile(r.name,"Linear",r.num_params,r.sensitivity,r.protected) for r in profiles_c]
        for e in plan:
            if e.protected: e.bits, e.note = "fp16", "protected"
        cand = [e for e in plan if not e.protected]
        n = len(cand)
        if not n: return plan
        n16 = max(1, round(n*self.fp16_pct/100))
        n8 = max(0, round(n*self.int8_pct/100))
        for i,e in enumerate(cand):
            if i<n16: e.bits,e.note = "fp16", f"high sens rank {i+1}"
            elif i<n16+n8: e.bits,e.note = "int8", "medium sens"
            else: e.bits,e.note = "int4", "low sens"
        return plan

    def build_uniform(self, profiles_c, bits):
        return [PlanProfile(r.name,"Linear",r.num_params,r.sensitivity,r.protected,
            "fp16" if r.protected else f"int{bits}", "protected" if r.protected else f"uniform int{bits}")
            for r in profiles_c]

def build_quant_plan(pc): return QuantPlanBuilder().build_mixed(pc)
def build_uniform_plan(pc,b): return QuantPlanBuilder().build_uniform(pc,b)
LayerProfile = PlanProfile
print("QuantPlanBuilder OK")


#### Phase D — `GPTQApplier`

**What:** run GPTQ calibration per layer and swap `nn.Linear` → `GPTQLinear`.

**Why:** this is the only step that modifies the main `model`.

In [ ]:
class GPTQApplier:
    def __init__(self, calibrator: CalibrationSession, device=DEVICE):
        self.cal, self.device = calibrator, device

    def apply(self, model, plan, captures=None) -> list[PlanProfile]:
        todo = [p for p in plan if p.bits in ("int4","int8")]
        if not todo: print("Nothing to quantize."); return []
        layers = dict(LinearLayerScanner(model).layers())
        applied = []
        print(f"\nGPTQApplier: {len(todo)} layers\n")
        for i, entry in enumerate(todo, 1):
            if entry.name not in layers or not isinstance(layers[entry.name], nn.Linear): continue
            layer = layers[entry.name]
            nb = int(entry.bits.replace("int",""))
            print(f"  [{i}/{len(todo)}] {entry.name}  bits={entry.bits}")
            gq = GPTQQuantizer(layer, nb, GPTQ_BLOCK_SIZE, GPTQ_DAMPING)
            hs = self.cal.register_gptq_hooks({entry.name: gq})
            self.cal.run(MAX_CALIB_BATCHES)
            for h in hs: h.remove()
            ql = QuantLinearFactory.build(layer, gq.quantize()).to(self.device)
            entry.weight_mse = LinearMetrics.weight_mse(layer, ql)
            entry.output_mse = LinearMetrics.output_mse(layer, ql, captures.get(entry.name) if captures else None)
            entry.fp_bytes = LinearMetrics.num_params(layer)*2
            entry.q_bytes = ql.storage_bytes(); entry.applied = True
            ModelPatcher.replace(model, entry.name, ql)
            applied.append(entry)
            print(f"       → GPTQLinear  mse={entry.weight_mse:.2e}")
        return applied

def apply_quant_plan(m, pd, cap=None):
    return GPTQApplier(CalibrationSession(m,processor,image,OCR_PROMPT)).apply(m,pd,cap)

LayerProfile = PlanProfile
print("GPTQApplier OK")

**▶ Run — build the bit plan**

### What happens
- `QuantPlanBuilder` reads `profiles_c` sensitivity ranks.
- Assigns int4 / int8 / fp16 per layer.
- Pie chart shows parameter distribution.

### Why mixed precision (not uniform int4)
- Uniform int4 is what Phase A did — it lost OCR lines.
- Keeping top-sensitive layers at fp16 costs little size but saves a lot of quality.
- int8 middle tier is a sweet spot: ~2× smaller than fp16, much better than int4.


In [ ]:
# Stage 11 — build plan
builder = QuantPlanBuilder()
profiles_d = builder.build_mixed(profiles_c) if MIXED_PRECISION else builder.build_uniform(profiles_c, BITS)
from collections import Counter
bc = Counter(p.bits for p in profiles_d)
print("Bit plan:", dict(bc))
for p in profiles_d[:15]:
    print(f"  {p.bits:5} {p.name.split('.')[-1]:20}  {p.note}")
fig,ax=plt.subplots(figsize=(5,4))
ax.pie([sum(p.num_params for p in profiles_d if p.bits==b) for b in ("int4","int8","fp16")],
       labels=["int4","int8","fp16"], autopct="%1.0f%%"); ax.set_title("Param share by precision")
plt.show()
total_quant_params = sum(p.num_params for p in profiles_d)


**▶ Run — apply GPTQ to the main model**

### What happens
- For each layer in the plan (int4/int8): run GPTQ calibration, swap to `GPTQLinear`.
- fp16 layers are left untouched.
- **This modifies `model`** — the only step that does.

### Why GPTQ calibration per layer
- GPTQ collects the Hessian (input covariance) during OCR forwards.
- Quantizes column-by-column, propagating error to remaining columns.
- Result: much lower **output** error than naive RTN at the same bit width.


In [ ]:
# Stage 12 — apply GPTQ
cal = CalibrationSession(model, processor, image, OCR_PROMPT)
applier = GPTQApplier(cal)
n_quant = sum(1 for p in profiles_d if p.bits in ("int4","int8"))
n_fp16 = sum(1 for p in profiles_d if p.bits == "fp16")
print(f"Quantizing {n_quant} layers, keeping {n_fp16} in fp16")
profiles_d_applied = applier.apply(model, profiles_d, captures)
print(f"\nSwapped {len(profiles_d_applied)} layers to GPTQLinear")


**▶ Weight error chart**

### What happens
- Bar chart of weight MSE per GPTQ-quantized layer.
- Green = int4, orange = int8.

### Why check this
- Confirms GPTQ reconstructed weights well.
- int8 bars should be shorter (lower MSE) than int4 bars.
- Outlier spikes may indicate layers that should have been fp16.


In [ ]:
# Stage 12b — weight error chart
if profiles_d_applied:
    fig, ax = plt.subplots(figsize=(10,4))
    names = [p.name.split(".")[-1] for p in profiles_d_applied]
    colors = {"int4":"#27ae60","int8":"#f39c12"}
    ax.barh(names[::-1], [p.weight_mse for p in profiles_d_applied][::-1],
            color=[colors.get(p.bits,"gray") for p in profiles_d_applied][::-1])
    ax.set_xlabel("Weight MSE"); ax.set_title("GPTQ weight error per layer")
    plt.tight_layout(); plt.show()


### Step 13 — Run OCR on the GPTQ model

### What happens
- `FlorenceOCRDetector` runs on the **main `model`** (GPTQ-quantized in Step 12).
- Draws bounding boxes on the calibration image.

### Why on main `model` (not `model_naive`)
- `model_naive` was the Phase A throwaway — blind int4 everything.
- `model` now has the smart mixed-precision GPTQ plan from Phase D.
- This is the **real** quality test.


In [ ]:
detector = FlorenceOCRDetector(processor)
lines, infer_s = detector.detect(image, model, DEVICE)
print(f"GPTQ model: {len(lines)} lines in {infer_s:.2f}s")
for line in lines[:6]:
    print(f"  • {line['text'][:65]}")
vis = image.copy(); draw = ImageDraw.Draw(vis)
for line in lines:
    b = line["bbox"]; draw.rectangle(b, outline="lime", width=2)
plt.figure(figsize=(8,6)); plt.imshow(vis)
plt.title(f"GPTQ detect — {len(lines)} lines"); plt.axis("off"); plt.show()


### Step 14 — Final scorecard

### What happens
Three-way comparison:

| Approach | Meaning |
|----------|---------|
| **fp16** | Original full-precision — the gold standard |
| **naive int4** | Phase A — everything int4, no planning |
| **GPTQ mixed** | Phase D — smart int4/int8/fp16 per layer |

### Why three-way (not just fp16 vs GPTQ)
- fp16 alone does not prove GPTQ is worth the effort.
- naive int4 shows the **floor** — GPTQ must beat this to justify the pipeline.
- If GPTQ ≈ fp16 lines but ~3× smaller → deployment win.

### Success criteria
- GPTQ line count **≥** naive int4 line count.
- Ideally within 1–2 lines of fp16.


In [ ]:
gptq_lines = len(lines)
ref = baseline_ocr
print(f"{'Approach':<14} {'Lines':>6} {'Δ fp16':>8} {'Time':>8}")
print("-"*40)
print(f"{'fp16':<14} {ref['fp16_lines']:>6} {'—':>8} {ref['fp16_infer_s']:>7.2f}s")
print(f"{'naive int4':<14} {ref['naive_lines']:>6} {ref['naive_lines']-ref['fp16_lines']:>+8d} {ref['naive_infer_s']:>7.2f}s")
print(f"{'GPTQ mixed':<14} {gptq_lines:>6} {gptq_lines-ref['fp16_lines']:>+8d} {infer_s:>7.2f}s")
fig, ax = plt.subplots(figsize=(5,3))
ax.bar(["fp16","naive int4","GPTQ"], [ref["fp16_lines"],ref["naive_lines"],gptq_lines],
       color=["#3498db","#e74c3c","#27ae60"])
ax.set_ylabel("Detected lines"); ax.set_title("Phase A naive vs Phase D GPTQ")
plt.tight_layout(); plt.show()
if gptq_lines >= ref["naive_lines"]:
    print("\n✓ GPTQ matched or beat naive baseline on line count.")


### Step 15 — GPTQ vs RTN on one layer (optional)

### What happens
- Picks one layer from Phase C captures.
- Compares RTN output MSE vs GPTQ output MSE side by side.

### Why this matters
- **RTN** only looks at weights — ignores what the layer actually sees during OCR.
- **GPTQ** uses calibration activations (Hessian) to minimize output error.
- GPTQ should win by a large margin — this is *why* we use GPTQ instead of RTN in Phase D.


In [ ]:
# Stage 15 — GPTQ vs RTN on one layer
cands = [(n, m) for n, m in LinearLayerScanner(model).layers(8)
         if m.weight.numel() >= MIN_PARAMS_TO_QUANT]
if cands and captures:
    dn, dl = cands[0]
    xin = captures.get(dn)
    if xin is not None and xin.numel():
        cmp = QuantMethodComparator(CalibrationSession(model, processor, image, OCR_PROMPT))
        mr, mg = cmp.rtn_mse(dl, xin), cmp.gptq_mse(dl, xin)
        print(f"Layer: {dn}")
        print(f"  RTN  output MSE: {mr:.4e}")
        print(f"  GPTQ output MSE: {mg:.4e}")
        print(f"  GPTQ wins by {(1 - mg / max(mr, 1e-12)) * 100:.1f}%")
else:
    print("Run Phase C first — need captures from sensitivity step.")


## Recap — what you built

### The pipeline
```
Phase A   NaiveQuantBaseline     →  baseline_ocr     (worst-case floor)
Phase B   LinearInventory        →  profiles_b       (who is big / protected)
Phase C   SensitivityAnalyzer    →  profiles_c       (who is fragile)
Phase D   QuantPlanBuilder       →  profiles_d       (int4 / int8 / fp16 plan)
          GPTQApplier            →  GPTQ model       (beat baseline ✓)
```

### Key takeaways
- **Don't quantize blindly** — inventory → sensitivity → mixed-precision GPTQ.
- **Only `nn.Linear`** — LayerNorm and embeddings stay fp16.
- **OCR line count** is the real quality metric, not weight MSE alone.
- **GPTQ > RTN** because it uses task-specific calibration data.

**Next:** [03 — Mobile Deployment](03_ocr_pipeline_mobile.ipynb) — pack these int4 weights for on-device inference.
